## PCM_pix — clean runner notebook

Этот ноутбук — **тонкий сценарий**:
- грузит данные из `data/`
- грузит/обучает суррогаты (legacy или new)
- оценивает качество суррогатов
- запускает оптимизацию (PSO / DE / PSO→DE)
- сохраняет результаты в `outputs/<run_name>/`

Старые большие ноутбуки остаются как архив (`3rd art_...`, `to_server_arch-...`).

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# --- CONFIG (меняй здесь) ---
CFG = {
    "run_name": "2026_03_12",
    "data_dir": "data",

    # dataset
    "wl": 1.55e-6,
    "mesh_names": [
        "Sb2Se3 - amorphous_mesh_sbse_2025.txt",
        "Sb2Se3 - crystal_mesh12_sbse_2025.txt",
    ],

    # surrogate
    "surrogate_mode": "legacy",   # "legacy" | "new"
    "device": "cpu",
    "epochs": 10000,
    "lr": 1e-3,

    # optimization settings
    "Nn": 11,
    "b_min_m": 50e-9,

    "opt_mode": "hybrid_pso_de",      # "preset" | "pso" | "pso_until" | "de_full" | "hybrid_pso_de"
    "preset_name": "pos_server_2026_03_04_night",

    # hyperopt modes
    # - legacy ключ (общий), оставлен для совместимости
    # - новые ключи позволяют управлять PSO и DE независимо
    #"hyperopt_mode": "run",        # legacy: "auto" | "use_saved" | "run"
    "pso_hyperopt_mode": "use_saved",    # "auto" | "use_saved" | "run"
    "de_hyperopt_mode": "use_saved",     # "auto" | "use_saved" | "run"

    # PSO
    "pso_threshold": 4,
    "pso_max_restarts": 20,
    "pso_n_particles": 3000,
    "pso_iters": 2000,
    "pso_c1": 0.5,
    "pso_c2": 0.3,
    "pso_w": 0.9,

    # DE (full compatibility)
    "de_init_mode": "init_ar",    # "init_ar" | "x0"
    "de_init_N": 10000,
    "de_mutation": 0.99,
    "de_recombination": 0.1,
    "de_maxiter": 10000,
    "de_popsize": 500,
    "de_tol": 1e-12,
    "de_atol": 1e-12,
    "de_polish": True,
    "de_callback_every": 1000,
    "de_use_linear_constraint": False,


}

# optional: reproducibility seed
SEED = 42
np.random.seed(SEED)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [33]:
from pcm_pix.run import start_run, save_json

run = start_run(outputs_dir="outputs", run_name=CFG["run_name"])
logger = run.logger

save_json(run.results / "config.json", CFG)
logger.info("main_clean started")

2026-03-12 12:41:45,575 | INFO | run_dir=/media/slava/Data/git/PCM_pix/outputs/2026_03_12
2026-03-12 12:41:45,577 | INFO | main_clean started


In [34]:
# --- Build dataset for surrogates ---
from pcm_pix.features import load_mesh_tables, make_nn_dataset

mesh_tables = load_mesh_tables(CFG, base_dir=CFG["data_dir"])
ds = make_nn_dataset(mesh_tables, wl=CFG["wl"])

logger.info("dataset: df=%s data_0=%s data_1=%s", len(ds.df), len(ds.data_0), len(ds.data_1))
logger.info("X_0=%s y_0=%s | X_1=%s y_1=%s", ds.X_0.shape, ds.y_0.shape, ds.X_1.shape, ds.y_1.shape)

2026-03-12 12:41:49,377 | INFO | dataset: df=98390 data_0=60360 data_1=38030
2026-03-12 12:41:49,378 | INFO | X_0=(60360, 3) y_0=(60360, 4) | X_1=(38030, 3) y_1=(38030, 4)


In [35]:
# --- Load / train surrogates ---
from pcm_pix.nn_surrogate import Net, get_device, load_legacy_surrogates, train_or_load_surrogates

# keep legacy unpickle working (torch.load(model) may reference __main__.Net)
Net

if CFG["device"] == "auto":
    CFG["device"] = get_device()

if CFG["surrogate_mode"] == "legacy":
    CFG["legacy_dir"] = CFG["data_dir"]
    sur0, sur1 = load_legacy_surrogates(CFG, device=CFG["device"])
    logger.info("surrogates loaded: legacy")
else:
    sur0, sur1 = train_or_load_surrogates(ds, run, CFG, force_train=False)
    logger.info("surrogates loaded: new-format")

/media/slava/Data/git/PCM_pix/.venv/lib/python3.12/site-packages/sklearn/base.py:463: InconsistentVersionWarning: Trying to unpickle estimator MinMaxScaler from version 1.6.1 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
2026-03-12 12:41:49,428 | INFO | surrogates loaded: legacy


In [36]:
# --- Evaluate surrogate quality ---
from pcm_pix.metrics import evaluate_surrogate

qa_am = evaluate_surrogate(ds.data_0, sur0, n=5000, seed=SEED, label="am")
qa_cr = evaluate_surrogate(ds.data_1, sur1, n=5000, seed=SEED, label="cr")

logger.info("QA am: %s", qa_am)
logger.info("QA cr: %s", qa_cr)
qa_am, qa_cr

2026-03-12 12:41:51,633 | INFO | QA am: {'label': 'am', 'n': 5000, 'R_mae': 0.0025238421934063887, 'R_rmse': 0.005561304011981201, 'T_mae': 0.003090210976341834, 'T_rmse': 0.007655419274479977, 'phiR_mae': 0.02427312091146617, 'phiR_rmse': 0.09400645956046887, 'phiT_mae': 0.014582478736734544, 'phiT_rmse': 0.11548982549827745, 'R_out_of_01': 14, 'T_out_of_01': 4}
2026-03-12 12:41:51,634 | INFO | QA cr: {'label': 'cr', 'n': 5000, 'R_mae': 0.0026156733580966597, 'R_rmse': 0.005520088491187868, 'T_mae': 0.002998169801017373, 'T_rmse': 0.0073268775979172494, 'phiR_mae': 0.023531035520791447, 'phiR_rmse': 0.09787895528330373, 'phiT_mae': 0.01618787454703999, 'phiT_rmse': 0.12370226249702683, 'R_out_of_01': 17, 'T_out_of_01': 1}


({'label': 'am',
  'n': 5000,
  'R_mae': 0.0025238421934063887,
  'R_rmse': 0.005561304011981201,
  'T_mae': 0.003090210976341834,
  'T_rmse': 0.007655419274479977,
  'phiR_mae': 0.02427312091146617,
  'phiR_rmse': 0.09400645956046887,
  'phiT_mae': 0.014582478736734544,
  'phiT_rmse': 0.11548982549827745,
  'R_out_of_01': 14,
  'T_out_of_01': 4},
 {'label': 'cr',
  'n': 5000,
  'R_mae': 0.0026156733580966597,
  'R_rmse': 0.005520088491187868,
  'T_mae': 0.002998169801017373,
  'T_rmse': 0.0073268775979172494,
  'phiR_mae': 0.023531035520791447,
  'phiR_rmse': 0.09787895528330373,
  'phiT_mae': 0.01618787454703999,
  'phiT_rmse': 0.12370226249702683,
  'R_out_of_01': 17,
  'T_out_of_01': 1})

In [37]:
# --- Presets / solutions catalog ---
from pcm_pix.optimize import save_solution, load_solution

PRESETS = {
    "pos_2026_03_03_example": np.array([
        999.7153881712717, 989.8568611314538, 846.9306855895525, 812.3405141712682,
        357.50476722603656, 999.0189388651343, 853.5246592828234, 979.0165540829283,
        952.2077494627958, 998.2131906719599, 998.0020446893369, 638.3004244094977,
        645.2087664126163, 681.6157122198844, 702.1334382560799, 284.8875621006104,
        882.7549390819086, 520.3952331766168, 442.78456983718775, 875.627636396043,
        683.0421367034168, 725.5677666087739, 305.7428105275045, 302.3913087111928,
        246.68691437766532, 261.130662543832, 25.83901557205877, 397.619402446743,
        17.140592766392103, 140.12532478580698, 735.0979445623718, 211.69540821165236,
        355.55785007186574, 4.522892213428676, 2.827729567363648,
        0.9073471746931481, 0.8273584342733484
    ]),
    "pos_server_2026_03_04_night":np.array([
        9.99613315e+02, 8.86918846e+02, 6.06035395e+02, 8.54046947e+02,
        5.17645193e+02, 8.79810042e+02, 5.65420119e+02, 9.64383427e+02,
        9.97814618e+02, 9.55635573e+02, 9.95016420e+02, 5.99598526e+02,
        8.29227746e+02, 5.39970803e+02, 8.04025215e+02, 4.17264700e+02,
        6.15239459e+02, 3.62347946e+02, 4.20726459e+02, 6.09650053e+02,
        2.69023666e+02, 4.13247590e+02, 3.76852495e+02, 5.40484452e+02,
        1.99690690e+02, 3.57938384e+02, 4.35299951e+01, 2.57512612e+02,
        2.45768992e+02, 2.64399901e+02, 5.09649663e+02, 1.68530071e+02,
        2.33417540e+01, 4.73217432e+00, 3.35695242e+00, 9.56090488e-01,
        8.91457136e-01
    ]),
        
}

# save presets once (idempotent)
for name, pos in PRESETS.items():
    save_solution(run, name=name, pos=pos, cost=None, meta={"source": "preset"})

logger.info("presets saved to %s", run.results / "solutions")

2026-03-12 12:42:09,037 | INFO | saved solution pos_2026_03_03_example.json
2026-03-12 12:42:09,039 | INFO | saved solution pos_server_2026_03_04_night.json
2026-03-12 12:42:09,040 | INFO | presets saved to outputs/2026_03_12/results/solutions


In [41]:
# --- PSO hyperopt: run or use saved ---
# Логика:
# - режим берём из CFG["pso_hyperopt_mode"], а если ключа нет — падаем обратно на legacy CFG["hyperopt_mode"]
# - результаты кэшируются в outputs/<run>/results/pso_*.csv
from pcm_pix.hyperopt import load_best_pso_hyperparams, apply_pso_hyperparams, run_pso_hyperopt

pso_mode = CFG.get("pso_hyperopt_mode", CFG.get("hyperopt_mode", "auto"))

if pso_mode == "run":
    # Make it faster by overriding these in CFG if needed
    # CFG["hyperopt_pso_trials"] = 80
    # CFG["hyperopt_pso_repeats"] = 2
    # CFG["hyperopt_pso_n_particles"] = 250
    # CFG["hyperopt_pso_iters"] = 60
    run_pso_hyperopt(sur0, sur1, CFG, run)

hp = load_best_pso_hyperparams(run.results)

if pso_mode == "use_saved" and hp is None:
    raise FileNotFoundError("pso_hyperopt_mode=use_saved but no pso_refine.csv / pso_random_search.csv")

if hp is not None and pso_mode in ("auto", "use_saved", "run"):
    apply_pso_hyperparams(CFG, hp)
    logger.info("PSO hyperparams loaded from %s: c1=%s c2=%s w=%s", hp.source, hp.c1, hp.c2, hp.w)
else:
    logger.info("PSO hyperparams unchanged (hp=%s mode=%s)", hp, pso_mode)

2026-03-12 12:45:39,781 | INFO | PSO hyperparams loaded from pso_refine.csv: c1=1.365595582786494 c2=1.3004900273877351 w=0.3332738612391048


In [42]:
# --- DE hyperopt: run or use saved ---
# Подбираем гиперпараметры SciPy differential_evolution:
# - de_mutation
# - de_recombination
# - de_popsize
#
# Результаты кэшируются в outputs/<run>/results/de_*.csv
from pcm_pix.hyperopt import load_best_de_hyperparams, apply_de_hyperparams, run_de_hyperopt

de_mode = CFG.get("de_hyperopt_mode", CFG.get("hyperopt_mode", "auto"))

# Важно: DE-hyperopt делает короткие прогоны DE и должен иметь стартовую точку (pos0)
# Берём её из preset (или можете подставить свой pos вручную).
pos0 = PRESETS[CFG["preset_name"]]

if de_mode == "run":
    # Make it faster by overriding these in CFG if needed
    # CFG["hyperopt_de_trials"] = 60
    # CFG["hyperopt_de_repeats"] = 2
    # CFG["hyperopt_de_maxiter"] = 50
    # CFG["hyperopt_de_polish"] = False
    run_de_hyperopt(sur0, sur1, CFG, run, pos0=pos0)

hp_de = load_best_de_hyperparams(run.results)

if de_mode == "use_saved" and hp_de is None:
    raise FileNotFoundError("de_hyperopt_mode=use_saved but no de_refine.csv / de_random_search.csv")

if hp_de is not None and de_mode in ("auto", "use_saved", "run"):
    apply_de_hyperparams(CFG, hp_de)
    logger.info(
        "DE hyperparams loaded from %s: mutation=%s recombination=%s popsize=%s",
        hp_de.source, hp_de.mutation, hp_de.recombination, hp_de.popsize,
    )
else:
    logger.info("DE hyperparams unchanged (hp_de=%s mode=%s)", hp_de, de_mode)

2026-03-12 12:45:39,945 | INFO | DE hyperparams loaded from de_refine.csv: mutation=0.2420622190083343 recombination=0.7001234155368052 popsize=54


In [ ]:
# --- Run optimization or use preset ---
from pcm_pix.optimize import (
    f_vec,
    run_pso,
    run_pso_until,
    run_de_full,
    run_hybrid_pso_de,
)

if CFG["opt_mode"] == "preset":
    pos = PRESETS[CFG["preset_name"]]
    cost = float(f_vec(pos.reshape(1, -1), sur0, sur1, CFG)[0])
    logger.info("using preset %s cost=%.6f", CFG["preset_name"], cost)

elif CFG["opt_mode"] == "pso":
    best = run_pso(sur0, sur1, CFG, run)
    pos, cost = best.pos, best.cost

elif CFG["opt_mode"] == "pso_until":
    best = run_pso_until(sur0, sur1, CFG, run)
    pos, cost = best.pos, best.cost

elif CFG["opt_mode"] == "de_full":
    # use preset (or set pos yourself)
    pos0 = PRESETS[CFG["preset_name"]]
    best = run_de_full(sur0, sur1, CFG, run, pos=pos0)
    pos, cost = best.pos, best.cost

else:
    best = run_hybrid_pso_de(sur0, sur1, CFG, run)
    pos, cost = best.pos, best.cost

logger.info("FINAL cost=%.6f", float(cost))
logger.info("FINAL pos_len=%s", len(pos))

pos, cost

2026-03-12 12:48:12,968 | INFO | PSO start: particles=3000 iters=3000 dims=37
2026-03-12 12:48:12,969 - pyswarms.single.global_best - INFO - Optimize for 3000 iters with {'c1': 0.5, 'c2': 0.3, 'w': 0.9}
pyswarms.single.global_best:   0%|          |0/3000

pyswarms.single.global_best: 100%|██████████|3000/3000, best_cost=3.82
2026-03-12 12:52:57,994 - pyswarms.single.global_best - INFO - Optimization finished | best cost: 3.8240325383317844, best pos: [6.96802206e+02 6.71862570e+02 5.70044226e+02 7.67893389e+02
 5.64573793e+02 5.65600763e+02 5.72535299e+02 6.61255389e+02
 9.77687797e+02 8.32185614e+02 9.38063233e+02 6.34132466e+02
 5.80925243e+02 5.05290797e+02 6.87437790e+02 4.62933121e+02
 4.29177446e+02 3.67210304e+02 4.69506021e+02 6.78770341e+02
 4.34487215e+02 8.46751867e+02 9.28830831e+01 2.00898638e+02
 1.31324391e+02 2.78727359e+02 2.80735905e+02 3.05603378e+02
 2.39536229e+02 3.38162167e+02 5.66327836e+02 3.12768651e+02
 6.59586568e+02 6.25699927e+00 3.37899216e+00 8.98572254e-01
 9.07540440e-01]
2026-03-12 12:52:57,995 | INFO | PSO done: cost=3.8240325383317844
2026-03-12 12:52:58,338 | INFO | DE start (full): maxiter=10000 popsize=500 init_mode=init_ar lc=False
2026-03-12 12:53:00,243 | INFO | Conv 0 3.931869
2026-03-12 12:53

In [ ]:
CFG

In [ ]:
# --- Save final solution to catalog ---
from pcm_pix.optimize import save_solution

name = f"final_{CFG['opt_mode']}"
path = save_solution(run, name=name, pos=pos, cost=float(cost), meta={"cfg": {"opt_mode": CFG["opt_mode"]}})
logger.info("final solution saved: %s", path.name)
path

In [ ]:
Из интересущийх меня моментов: 
1) я получил такой вывод 2026-03-12 12:48:12,969 - pyswarms.single.global_best - INFO - Optimize for 3000 iters with {'c1': 0.5, 'c2': 0.3, 'w': 0.9}
Это говорит, что гиперпараметры либо не поменялись после гипероптимизации, либо не читается CFG - если параметры были изменены в результате гипероптимизации - это должно быть учтено
2) у DE оптимизации не выводится никаких сообщений о ходе оптимизации - исправить
